# 🚀 L&D Designs — Full Automated Outreach Pipeline

**What this does:**
- Emails all leads that have an email address
- Searches the web to find emails for businesses that don't have one
- Sends SMS to all 145 mobile numbers via Twilio
- Automatically follows up after 3 days to non-responders
- Logs everything so nothing gets double-sent

**Run order:** Cell 1 → 2 → 3 → 4 → 5 → 6

---
> To send follow-ups, come back in 3 days and run Cell 6 only.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'twilio', 'requests', 'beautifulsoup4'])
print('Ready.')

In [ ]:
# ── Fill in your details here ──────────────────────────────────────────────

# Gmail (free) — setup: myaccount.google.com → Security → App passwords
GMAIL_ADDRESS      = 'your@gmail.com'
GMAIL_APP_PASSWORD = 'xxxx xxxx xxxx xxxx'

# Twilio (for SMS, ~£4 total) — get from console.twilio.com
TWILIO_ACCOUNT_SID = 'ACxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx'
TWILIO_AUTH_TOKEN  = 'xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx'
TWILIO_FROM_NUMBER = ''  # your Twilio number e.g. +441234567890

# Your details
YOUR_NAME     = 'Dylan'
YOUR_PHONE    = '07301 181878'
SHOWCASE_LINK = 'https://mellow-speculoos-d1850c.netlify.app/showcase.html'

SEND_EMAILS = GMAIL_ADDRESS != 'your@gmail.com'
SEND_SMS    = bool(TWILIO_FROM_NUMBER)
print('Emails:', 'ENABLED' if SEND_EMAILS else 'DISABLED — fill in Gmail details')
print('SMS   :', 'ENABLED' if SEND_SMS else 'DISABLED — fill in Twilio number when ready')

In [ ]:
import csv, re, json, os
from datetime import datetime, timedelta

try:
    from google.colab import files
    print('Upload your leads.csv...')
    uploaded = files.upload()
    csv_file = list(uploaded.keys())[0]
except:
    csv_file = 'leads.csv'

with open(csv_file, newline='', encoding='utf-8') as f:
    leads = list(csv.DictReader(f))

LOG_FILE = 'send_log.json'
send_log = json.load(open(LOG_FILE)) if os.path.exists(LOG_FILE) else {}

def already_sent(lid, mtype): return send_log.get(lid, {}).get(mtype) is not None
def mark_sent(lid, mtype, status='sent'):
    if lid not in send_log: send_log[lid] = {}
    send_log[lid][mtype] = {'status': status, 'time': datetime.now().isoformat()}
    json.dump(send_log, open(LOG_FILE,'w'), indent=2)

email_leads  = [l for l in leads if l.get('email') and '@' in l['email']]
mobile_leads = [l for l in leads if l.get('is_mobile') == 'True']
unsent_email = [l for l in email_leads  if not already_sent(l['id'], 'email')]
unsent_sms   = [l for l in mobile_leads if not already_sent(l['id'], 'sms')]

print(f'Total leads    : {len(leads)}')
print(f'Emailable      : {len(email_leads)} ({len(unsent_email)} unsent)')
print(f'Mobile SMS     : {len(mobile_leads)} ({len(unsent_sms)} unsent)')

In [ ]:
# Optional: search Google to find emails for businesses that don't have one
# Change MAX_SEARCH — each search takes ~3 seconds
import requests, time, random

EMAIL_RE = re.compile(r'[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}')
SKIP = {'gmail.com','yahoo.com','hotmail.com','example.com','lddesigns',
        'sentry.io','wixpress.com','googletagmanager','facebook.com'}

def find_email(name, address):
    try:
        r = requests.get('https://www.google.com/search',
            params={'q': f'{name} {address} contact email', 'num': 5},
            headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
        for e in EMAIL_RE.findall(r.text):
            if not any(s in e.lower() for s in SKIP):
                return e
    except: pass
    return None

MAX_SEARCH = 50  # increase to search more — takes ~2.5 mins per 50
no_email = [l for l in leads if not l.get('email') and l.get('phone')]
print(f'Searching for emails for {min(MAX_SEARCH, len(no_email))} businesses...')
found = 0
for i, lead in enumerate(no_email[:MAX_SEARCH]):
    email = find_email(lead['name'], lead['address'])
    if email:
        lead['email'] = email
        found += 1
        print(f'  Found: {lead["name"]} → {email}')
    else:
        print(f'  [{i+1}] No email: {lead["name"]}')
    time.sleep(random.uniform(2, 4))

with open(csv_file, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=leads[0].keys())
    w.writeheader(); w.writerows(leads)

email_leads  = [l for l in leads if l.get('email') and '@' in l['email']]
unsent_email = [l for l in email_leads if not already_sent(l['id'], 'email')]
print(f'\nFound {found} new emails. Total emailable: {len(email_leads)} ({len(unsent_email)} unsent)')

In [ ]:
import smtplib, time, random
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

EMAIL_SUBJECT = 'Quick question about {name}'
EMAIL_BODY = """Hi,

I noticed {name} doesn't have a website yet.

I build professional websites for local businesses in Wigan from just £199 — usually done within a week, no monthly fees.

Here's some of my recent work: {showcase}

Free quote? Just reply or WhatsApp {phone}.

Thanks, {your_name}\nL&D Designs"""

SMS_BODY = ("Hi, I noticed {name} doesn't have a website. "
            "I build professional sites for Wigan businesses from £199 — "
            "done in days, no monthly fees. Free quote? — {your_name}, L&D Designs")

email_leads  = [l for l in leads if l.get('email') and '@' in l['email']]
mobile_leads = [l for l in leads if l.get('is_mobile') == 'True']
unsent_email = [l for l in email_leads  if not already_sent(l['id'], 'email')]
unsent_sms   = [l for l in mobile_leads if not already_sent(l['id'], 'sms')]

print(f'Emails to send : {len(unsent_email) if SEND_EMAILS else 0}')
print(f'SMS to send    : {len(unsent_sms)   if SEND_SMS   else 0}')

if input('Type YES to send: ').strip() == 'YES':
    es = ef = ss = sf = 0

    if SEND_EMAILS and unsent_email:
        srv = smtplib.SMTP_SSL('smtp.gmail.com', 465)
        srv.login(GMAIL_ADDRESS, GMAIL_APP_PASSWORD)
        for i,l in enumerate(unsent_email):
            try:
                msg = MIMEMultipart()
                msg['From'] = GMAIL_ADDRESS; msg['To'] = l['email']
                msg['Subject'] = EMAIL_SUBJECT.format(name=l['name'])
                msg.attach(MIMEText(EMAIL_BODY.format(name=l['name'],
                    showcase=SHOWCASE_LINK, phone=YOUR_PHONE, your_name=YOUR_NAME), 'plain'))
                srv.sendmail(GMAIL_ADDRESS, l['email'], msg.as_string())
                mark_sent(l['id'], 'email'); es += 1
                print(f'[{i+1}/{len(unsent_email)}] ✓ {l["name"]} → {l["email"]}')
            except Exception as e:
                mark_sent(l['id'], 'email', f'failed:{e}'); ef += 1
                print(f'[{i+1}] ✗ {l["name"]} — {e}')
            if i < len(unsent_email)-1: time.sleep(random.uniform(3,6))
        srv.quit()
        print(f'Emails: {es} sent, {ef} failed')

    if SEND_SMS and unsent_sms:
        from twilio.rest import Client
        cli = Client(TWILIO_ACCOUNT_SID, TWILIO_AUTH_TOKEN)
        for i,l in enumerate(unsent_sms):
            try:
                cli.messages.create(body=SMS_BODY.format(name=l['name'],your_name=YOUR_NAME),
                    from_=TWILIO_FROM_NUMBER, to=l['phone_e164'])
                mark_sent(l['id'], 'sms'); ss += 1
                print(f'[{i+1}/{len(unsent_sms)}] ✓ SMS {l["name"]} → {l["phone_e164"]}')
            except Exception as e:
                mark_sent(l['id'], 'sms', f'failed:{e}'); sf += 1
                print(f'[{i+1}] ✗ {l["name"]} — {e}')
            if i < len(unsent_sms)-1: time.sleep(random.uniform(1,2))
        print(f'SMS: {ss} sent, {sf} failed')

    print('\n✅ Done. Run Cell 6 in 3 days for follow-ups.')

In [ ]:
# ── Follow-ups — run this 3 days after Cell 5 ─────────────────────────────
import smtplib, time, random
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from datetime import datetime, timedelta

FOLLOWUP_SUBJECT = 'Re: Quick question about {name}'
FOLLOWUP_BODY = """Hi again,

Just following up on my message from a few days ago about building a website for {name}.

If you're interested or have any questions, just reply here or WhatsApp {phone}.

No pressure — just wanted to make sure it didn't get buried!

Thanks, {your_name}\nL&D Designs"""

cutoff = datetime.now() - timedelta(days=3)
targets = []
for l in leads:
    log = send_log.get(l['id'], {}).get('email', {})
    if log.get('status') == 'sent' and not already_sent(l['id'], 'followup'):
        if datetime.fromisoformat(log['time']) <= cutoff:
            targets.append(l)

print(f'Follow-ups ready: {len(targets)}')
if not targets:
    print('None ready yet — come back 3 days after your initial send.')
elif input(f'Send {len(targets)} follow-ups? Type YES: ').strip() == 'YES':
    srv = smtplib.SMTP_SSL('smtp.gmail.com', 465)
    srv.login(GMAIL_ADDRESS, GMAIL_APP_PASSWORD)
    sent = 0
    for i,l in enumerate(targets):
        try:
            msg = MIMEMultipart()
            msg['From'] = GMAIL_ADDRESS; msg['To'] = l['email']
            msg['Subject'] = FOLLOWUP_SUBJECT.format(name=l['name'])
            msg.attach(MIMEText(FOLLOWUP_BODY.format(
                name=l['name'], phone=YOUR_PHONE, your_name=YOUR_NAME), 'plain'))
            srv.sendmail(GMAIL_ADDRESS, l['email'], msg.as_string())
            mark_sent(l['id'], 'followup'); sent += 1
            print(f'[{i+1}/{len(targets)}] ✓ {l["name"]}')
        except Exception as e:
            print(f'[{i+1}] ✗ {l["name"]} — {e}')
        time.sleep(random.uniform(3,5))
    srv.quit()
    print(f'\n✅ {sent} follow-ups sent.')